# 102 — Voice Pipeline: TTS + Whisper + Agent
## Build an end-to-end spoken AI assistant with LangGraph
⏱ ~45 min

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Esturban/agent/blob/master/examples/102-voice-pipeline/voice_pipeline_workbook.ipynb)

Voice interfaces are reshaping how people interact with AI. This workshop builds a **complete voice pipeline** — from audio in to audio out — using only OpenAI cloud APIs. No local model downloads, no GPU required.

The pipeline has three stages wired as a LangGraph graph:
1. **STT (Speech-to-Text)** — OpenAI Whisper transcribes spoken audio to text
2. **LLM** — GPT-4o-mini generates a text response
3. **TTS (Text-to-Speech)** — OpenAI TTS converts the text response back to audio

This architecture makes **latency explicit and measurable** at each stage — a prerequisite for production TTFA (Time-To-First-Audio) optimisation.

---
### Workshop Roadmap
| # | Topic |
|---|-------|
| 1 | **Concepts** — Voice pipelines, STT/TTS APIs, TTFA |
| 2 | **Setup** — Install deps, configure API key |
| 3 | **TTS** — Text → Audio with OpenAI tts-1 |
| 4 | **STT** — Audio → Text with OpenAI Whisper |
| 5 | **LLM Node** — Text → Text with GPT-4o-mini |
| 6 | **LangGraph State** — Typed state for the pipeline |
| 7 | **Full Pipeline** — Wire all three nodes into a graph |
| 8 | **Latency Profiling** — Measure each stage |
| ★ | **Exercises + Answer Key** |

---
### Prerequisites
- Python 3.10+, or Google Colab (free tier works)
- `OPENAI_API_KEY` in `.env` or Colab Secrets
- No local GPU needed — everything is cloud API calls

### Key References
> OpenAI (2022). *Robust Speech Recognition via Large-Scale Weak Supervision* (Whisper). [arxiv:2212.04356](https://arxiv.org/abs/2212.04356)
>
> OpenAI TTS API docs: https://platform.openai.com/docs/guides/text-to-speech
>
> LangGraph docs: https://langchain-ai.github.io/langgraph/

In [ ]:
import sys

def _in_colab():
    try:
        import google.colab
        return True
    except ImportError:
        return False

if _in_colab():
    import subprocess
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q",
         "openai", "langchain-openai", "langgraph", "langchain-core", "python-dotenv"],
        check=True
    )
    print("Colab install complete.")
else:
    print("Local environment — skipping install.")

In [ ]:
import os

try:
    from google.colab import userdata
    os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
except ImportError:
    from dotenv import load_dotenv
    load_dotenv()

key = os.environ.get("OPENAI_API_KEY", "")
if not (bool(key) and key.startswith('sk-')):
    raise RuntimeError("OPENAI_API_KEY is required for the live voice-pipeline cells.")
print("API key ready: True")

---
## Part 1 — Concepts: What is a Voice Pipeline?

A voice pipeline converts spoken audio into intelligent spoken responses. The canonical architecture is:

```
┌─────────────┐     ┌─────────────┐     ┌─────────────┐     ┌─────────────┐
│  Input      │     │    STT      │     │    LLM      │     │    TTS      │
│  Audio      │────►│  (Whisper)  │────►│  (GPT-4o)   │────►│  (tts-1)   │
│  .mp3/.wav  │     │  → text     │     │  → answer   │     │  → audio   │
└─────────────┘     └─────────────┘     └─────────────┘     └─────────────┘
```

### Why model it as a graph?

Each stage has a distinct latency profile:

| Stage | Typical latency | Bottleneck |
|-------|----------------|------------|
| STT (Whisper) | 200–800 ms | Audio length, network |
| LLM (GPT-4o) | 500–3000 ms | Response length, load |
| TTS (tts-1) | 300–1500 ms | Response length, network |

Making them **separate named nodes** in LangGraph means:
- You can time each stage independently
- You can add retries or fallbacks per node
- You can swap any stage (e.g. Whisper → Deepgram) without touching the others

### TTFA — Time To First Audio

The key production metric. In the naive pipeline, TTS only starts after the LLM finishes. A more advanced approach:

1. Stream LLM tokens
2. Buffer until a sentence boundary (`.`, `!`, `?`)
3. Send that sentence to TTS immediately
4. Continue while TTS speaks the first sentence

This workshop builds the **serial baseline**. Part 8 shows where to insert streaming.

---
## Part 2 — Setup: Model Constants

The pipeline uses three OpenAI models, all covered by a single `OPENAI_API_KEY`:

| Constant | Value | Purpose |
|----------|-------|---------|
| `MODEL_LLM` | `gpt-4o-mini` | Fast, cheap LLM for Q&A |
| `MODEL_STT` | `whisper-1` | OpenAI Whisper cloud STT |
| `MODEL_TTS` | `tts-1` | OpenAI text-to-speech |
| `TTS_VOICE` | `nova` | One of 6 available voices |

Available TTS voices: `alloy`, `echo`, `fable`, `onyx`, `nova`, `shimmer`

In [ ]:
from openai import OpenAI

# Model identifiers — match src/tools.py exactly
MODEL_LLM = "gpt-5.4-nano"
MODEL_STT = "whisper-1"   # OpenAI Whisper cloud endpoint
MODEL_TTS = "tts-1"       # OpenAI TTS endpoint
TTS_VOICE = "nova"

# Demo question used throughout the workshop
DEMO_QUESTION = "What are the three most important things to know about building LangGraph agents?"

# Instantiate client once — reuses across cells
client = OpenAI()

print(f"LLM  : {MODEL_LLM}")
print(f"STT  : {MODEL_STT}")
print(f"TTS  : {MODEL_TTS} (voice: {TTS_VOICE})")
print(f"Demo : {DEMO_QUESTION}")

---
## Part 3 — TTS: Text → Audio

OpenAI's TTS API (`tts-1`) converts text to spoken audio in a single REST call.

### API anatomy

```python
resp = client.audio.speech.create(
    model="tts-1",       # or "tts-1-hd" for higher quality
    voice="nova",        # one of 6 voices
    input="Hello world"  # text to speak
)
resp.stream_to_file("out.mp3")  # saves to disk
```

### Why `tts-1` vs `tts-1-hd`?
- `tts-1`: lower latency, good for real-time applications
- `tts-1-hd`: higher fidelity, better for recorded content

For a voice assistant where TTFA matters, **`tts-1` is the right choice**.

In [ ]:
import tempfile
import os

def text_to_speech(text: str, out_path: str) -> str:
    """Call OpenAI TTS and write the mp3 to out_path; return the path."""
    client = OpenAI()
    resp = client.audio.speech.create(model=MODEL_TTS, voice=TTS_VOICE, input=text)
    resp.stream_to_file(out_path)
    return out_path

# Generate a short test phrase to verify TTS works
test_phrase = "Voice pipeline initialized. Ready to assist."
tts_test_path = tempfile.mktemp(suffix=".mp3")

text_to_speech(test_phrase, tts_test_path)
size_kb = os.path.getsize(tts_test_path) // 1024
print(f"TTS output: {tts_test_path}")
print(f"File size : {size_kb} KB")
print(f"Text      : '{test_phrase}'")

In [ ]:
# Now generate the demo question audio — this becomes the pipeline's input
audio_in = tempfile.mktemp(suffix=".mp3")
text_to_speech(DEMO_QUESTION, audio_in)

size_kb = os.path.getsize(audio_in) // 1024
print(f"Input audio  : {audio_in} ({size_kb} KB)")
print(f"Question text: {DEMO_QUESTION}")
print()
print("The demo is self-contained: TTS creates the 'user question' audio")
print("so we can test the full pipeline without recording a microphone.")

---
## Part 4 — STT: Audio → Text with Whisper

**Whisper** (2022) is OpenAI's multilingual speech recognition model trained on 680,000 hours of web audio. The cloud API (`whisper-1`) handles the heavy lifting — you just send a file.

### How Whisper works (simplified)

```
Audio waveform
    │
    ▼
Log-Mel Spectrogram  ← 30-second chunks, 80-channel
    │
    ▼
Encoder (Transformer)  ← learns acoustic representations
    │
    ▼
Decoder (Transformer)  ← auto-regressive text generation
    │
    ▼
Transcript
```

### API anatomy

```python
with open("audio.mp3", "rb") as f:
    resp = client.audio.transcriptions.create(
        model="whisper-1",
        file=f
    )
print(resp.text)  # the transcript string
```

Supported formats: `mp3`, `mp4`, `mpeg`, `mpga`, `m4a`, `wav`, `webm`

In [ ]:
def speech_to_text(audio_path: str) -> str:
    """Transcribe an audio file via OpenAI Whisper; return the transcript string."""
    client = OpenAI()
    with open(audio_path, "rb") as f:
        resp = client.audio.transcriptions.create(model=MODEL_STT, file=f)
    return resp.text

# Transcribe the audio we just generated
print("Sending audio to Whisper...")
transcript = speech_to_text(audio_in)

print(f"Original  : {DEMO_QUESTION}")
print(f"Transcript: {transcript}")
print()

# Check round-trip fidelity
match = transcript.strip().lower() == DEMO_QUESTION.strip().lower()
print(f"Exact match: {match}")
print("(Minor punctuation differences are normal — Whisper may add/drop periods)")

---
## Part 5 — LLM Node: Text → Text

The middle node takes the transcript and produces a text answer using GPT-4o-mini via LangChain's `ChatOpenAI`.

### Why `ChatOpenAI` instead of raw `openai.chat.completions`?

LangGraph works best when nodes return values compatible with the LangChain message format. `ChatOpenAI` gives us:
- Automatic message formatting
- Drop-in compatibility with other LangChain LLMs (swap Anthropic, Groq, etc. in one line)
- Streaming support without extra wiring

### Pattern

```python
from langchain_core.messages import HumanMessage
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-5.4-nano")
answer = llm.invoke([HumanMessage(content="your question")]).content
```

In [ ]:
from langchain_core.messages import HumanMessage
from langchain_openai import ChatOpenAI

# Test the LLM node in isolation
llm = ChatOpenAI(model=MODEL_LLM)

print(f"Question: {transcript}")
print()
print("Generating response...")
response_text = llm.invoke([HumanMessage(content=transcript)]).content

print(f"Response ({len(response_text)} chars):")
print("-" * 60)
print(response_text[:500])
if len(response_text) > 500:
    print(f"... [{len(response_text) - 500} more chars]")

---
## Part 6 — LangGraph State: Typed Pipeline State

LangGraph uses a **typed state dict** that flows through every node. Each node receives the current state and returns a partial update.

### VoiceState design

```
VoiceState
├── audio_in   : str   ← path to the input mp3
├── transcript : str   ← Whisper output (filled by transcribe node)
├── response   : str   ← LLM answer   (filled by respond node)
└── audio_out  : str   ← path to TTS output mp3 (filled by speak node)
```

### Why `TypedDict` instead of a dataclass?

LangGraph's `StateGraph` expects dict-compatible state. `TypedDict` gives you:
- Type annotations for IDE completion and mypy checking
- Dict semantics (node returns `dict`, graph merges it into state)
- No serialization overhead — it's just a plain dict at runtime

### Node contract

Every node must:
1. Accept `state: VoiceState` as its only argument
2. Return a **partial dict** with only the keys it updates

```python
def transcribe(state: VoiceState) -> dict:
    text = speech_to_text(state["audio_in"])
    return {"transcript": text}   # ← only returns what it changed
```

In [ ]:
from typing import TypedDict

class VoiceState(TypedDict):
    audio_in: str    # path to input mp3
    transcript: str  # Whisper output
    response: str    # LLM text answer
    audio_out: str   # path to TTS output mp3

# Show what an initial state looks like
initial_state: VoiceState = {
    "audio_in": audio_in,
    "transcript": "",
    "response": "",
    "audio_out": "",
}

print("Initial state:")
for k, v in initial_state.items():
    display_v = v if len(str(v)) < 80 else v[:77] + "..."
    print(f"  {k:<12}: {repr(display_v)}")

---
## Part 7 — Three Nodes: transcribe → respond → speak

Each node is a pure function with the same signature. Let's define them explicitly before wiring the graph.

### Node responsibilities

```
transcribe(state)
  reads  : state["audio_in"]
  writes : state["transcript"]
  calls  : OpenAI Whisper API

respond(state)
  reads  : state["transcript"]
  writes : state["response"]
  calls  : OpenAI Chat Completions (GPT-4o-mini)

speak(state)
  reads  : state["response"]
  writes : state["audio_out"]
  calls  : OpenAI TTS API
```

Each node is independently testable — you can unit-test `respond` by passing a dict with `transcript` already filled.

In [ ]:
import tempfile

def transcribe(state: VoiceState) -> dict:
    """Node 1: speech-to-text via Whisper."""
    text = speech_to_text(state["audio_in"])
    print(f"  [STT] {text}")
    return {"transcript": text}


def respond(state: VoiceState) -> dict:
    """Node 2: LLM answers the transcript."""
    llm = ChatOpenAI(model=MODEL_LLM)
    answer = llm.invoke([HumanMessage(content=state["transcript"])]).content
    return {"response": answer}


def speak(state: VoiceState) -> dict:
    """Node 3: text-to-speech via TTS."""
    out = tempfile.mktemp(suffix=".mp3")
    text_to_speech(state["response"], out)
    return {"audio_out": out}


print("Node functions defined:")
print("  transcribe(state) → {transcript}")
print("  respond(state)    → {response}")
print("  speak(state)      → {audio_out}")

---
## Part 8 — Building the LangGraph Pipeline

Now we wire the three nodes into a `StateGraph`. The graph is **linear** — no branching or conditions — because every voice query follows the same STT → LLM → TTS path.

### Graph structure

```
START
  │
  ▼
transcribe  (STT node)
  │
  ▼
respond     (LLM node)
  │
  ▼
speak       (TTS node)
  │
  ▼
END
```

### `create_workflow()` — the factory function

Encapsulating graph creation in a factory function (`create_workflow`) is a LangGraph convention. It means:
- The compiled graph is reusable (call `create_workflow()` once, invoke many times)
- Tests can import and compile the graph without running it
- You can pass config to the factory to create graph variants

In [ ]:
from langgraph.graph import END, START, StateGraph

def create_workflow():
    """Build and compile the three-node voice pipeline graph."""
    g = StateGraph(VoiceState)

    # Register nodes
    g.add_node("transcribe", transcribe)
    g.add_node("respond", respond)
    g.add_node("speak", speak)

    # Wire edges: linear chain
    g.add_edge(START, "transcribe")
    g.add_edge("transcribe", "respond")
    g.add_edge("respond", "speak")
    g.add_edge("speak", END)

    return g.compile()


# Compile once
app = create_workflow()
print("Graph compiled successfully.")
print()
print("Nodes:", list(app.get_graph().nodes.keys()))

---
## Part 9 — Running the Full Pipeline

With the graph compiled, a single `app.invoke(initial_state)` call drives the audio through all three stages and returns the final state.

The pipeline makes **three API calls** in sequence:
1. `POST /v1/audio/transcriptions` — Whisper
2. `POST /v1/chat/completions` — GPT-4o-mini
3. `POST /v1/audio/speech` — TTS

Total wall-clock time is the sum of all three plus network overhead.

In [ ]:
import time

print("Running voice pipeline (STT → LLM → TTS)...")
print("=" * 60)

t_start = time.time()

result = app.invoke({
    "audio_in": audio_in,
    "transcript": "",
    "response": "",
    "audio_out": "",
})

t_end = time.time()

print()
print(f"Total time  : {t_end - t_start:.2f}s")
print()
print(f"Transcript  : {result['transcript']}")
print()
print(f"Response    : {result['response'][:300]}")
if len(result['response']) > 300:
    print(f"              ... [{len(result['response']) - 300} more chars]")
print()

out_kb = os.path.getsize(result["audio_out"]) // 1024
print(f"Output audio: {result['audio_out']} ({out_kb} KB)")
print()
print("Play both mp3 files to hear the full round-trip:")
print(f"  Input  : {audio_in}")
print(f"  Output : {result['audio_out']}")

---
## Part 10 — Latency Profiling per Node

Understanding where time is spent is critical for production voice systems. A 3-second total latency breaks down very differently across the three stages depending on the audio length and response length.

### Why profile per node?

- **STT latency** scales with audio duration
- **LLM latency** scales with response length (tokens/sec throughput)
- **TTS latency** scales with text length

Knowing the bottleneck tells you where to optimise:
- STT bottleneck → shorter user prompts, local Whisper, parallel chunking
- LLM bottleneck → smaller model, streaming + early TTS start (TTFA)
- TTS bottleneck → cache repeated phrases, streaming TTS output

In [ ]:
import time

def timed_transcribe(state: VoiceState) -> dict:
    t0 = time.time()
    text = speech_to_text(state["audio_in"])
    dt = time.time() - t0
    print(f"  [STT]   {dt:.2f}s  → '{text[:60]}'")
    return {"transcript": text}


def timed_respond(state: VoiceState) -> dict:
    t0 = time.time()
    llm = ChatOpenAI(model=MODEL_LLM)
    answer = llm.invoke([HumanMessage(content=state["transcript"])]).content
    dt = time.time() - t0
    print(f"  [LLM]   {dt:.2f}s  → {len(answer)} chars")
    return {"response": answer}


def timed_speak(state: VoiceState) -> dict:
    t0 = time.time()
    out = tempfile.mktemp(suffix=".mp3")
    text_to_speech(state["response"], out)
    dt = time.time() - t0
    kb = os.path.getsize(out) // 1024
    print(f"  [TTS]   {dt:.2f}s  → {kb} KB mp3")
    return {"audio_out": out}


def create_timed_workflow():
    g = StateGraph(VoiceState)
    g.add_node("transcribe", timed_transcribe)
    g.add_node("respond", timed_respond)
    g.add_node("speak", timed_speak)
    g.add_edge(START, "transcribe")
    g.add_edge("transcribe", "respond")
    g.add_edge("respond", "speak")
    g.add_edge("speak", END)
    return g.compile()


print("Profiling run:")
print("-" * 60)

timed_app = create_timed_workflow()
t0_total = time.time()

timed_result = timed_app.invoke({
    "audio_in": audio_in,
    "transcript": "",
    "response": "",
    "audio_out": "",
})

total = time.time() - t0_total
print("-" * 60)
print(f"  [TOTAL] {total:.2f}s")

---
## Part 11 — TTFA: The Production Optimisation

In the baseline pipeline, TTS only starts after the **full LLM response** is ready. For a 500-word answer at 50 tokens/second, that's ~3 seconds of silence before the user hears anything.

### The TTFA pattern

```
BASELINE (serial):
  ┌──────┐    ┌──────────────┐    ┌──────────┐
  │ STT  │───►│  LLM (full)  │───►│ TTS (all)│
  └──────┘    └──────────────┘    └──────────┘
              ←── 3 seconds ───►  ←── 2s ───►
  User hears nothing for 5 seconds.

TTFA OPTIMISED (streaming):
  ┌──────┐    ┌────────────────────────────────┐
  │ STT  │───►│  LLM stream  → sentence buffer │
  └──────┘    │              │ TTS chunk 1     │
              │              │ TTS chunk 2     │
              │              │ TTS chunk 3 ... │
              └────────────────────────────────┘
  User hears first sentence in ~0.8 seconds.
```

### Implementation sketch

The code below shows the LLM streaming approach — collecting tokens until a sentence boundary, then calling TTS on each chunk. This is the production pattern used by voice assistants.

In [ ]:
import re

def stream_respond_and_speak(transcript: str) -> list[str]:
    """
    TTFA demonstration: stream LLM tokens, split on sentence boundaries,
    and call TTS on each sentence as soon as it's complete.

    Returns a list of mp3 paths (one per sentence).
    """
    llm = ChatOpenAI(model=MODEL_LLM, streaming=True)
    sentence_endings = re.compile(r'(?<=[.!?])\s+')

    audio_chunks = []
    buffer = ""
    chunk_index = 0

    print(f"  Streaming response (one TTS call per sentence):")

    for chunk in llm.stream([HumanMessage(content=transcript)]):
        token = chunk.content
        buffer += token

        # Check for sentence boundary
        parts = sentence_endings.split(buffer)
        if len(parts) > 1:
            # All complete sentences
            for sentence in parts[:-1]:
                sentence = sentence.strip()
                if not sentence:
                    continue
                t0 = time.time()
                out_path = tempfile.mktemp(suffix=".mp3")
                text_to_speech(sentence, out_path)
                dt = time.time() - t0
                kb = os.path.getsize(out_path) // 1024
                chunk_index += 1
                print(f"    chunk {chunk_index}: {dt:.2f}s → {kb} KB | '{sentence[:60]}'")
                audio_chunks.append(out_path)
            buffer = parts[-1]  # remainder after last complete sentence

    # Handle any remaining text
    if buffer.strip():
        out_path = tempfile.mktemp(suffix=".mp3")
        text_to_speech(buffer.strip(), out_path)
        audio_chunks.append(out_path)

    return audio_chunks


print("Running TTFA demo (streaming LLM + per-sentence TTS):")
print("=" * 60)
t0 = time.time()
chunks = stream_respond_and_speak(transcript)
print(f"\nTotal chunks: {len(chunks)}")
print(f"Total time  : {time.time() - t0:.2f}s")
print("First audio chunk was ready much sooner than the full response.")

---
## Part 12 — Switching Voices

OpenAI TTS supports 6 voices. Choosing the right voice for your product is a UX decision — some voices sound more authoritative, others more conversational.

| Voice | Character |
|-------|-----------|
| `alloy` | Neutral, balanced |
| `echo` | Slightly deeper, male |
| `fable` | British accent, storytelling |
| `onyx` | Deep, authoritative |
| `nova` | Clear, friendly (default) |
| `shimmer` | Warm, expressive |

The voice is just a string parameter — swapping it costs zero code changes in the rest of the pipeline.

In [ ]:
# Generate the same short phrase in two voices for comparison
sample_text = "Welcome to the voice pipeline workshop. Let's build something great."

VOICES = ["nova", "onyx"]  # keeping to 2 API calls for cost awareness

voice_paths = {}
for voice in VOICES:
    out_path = tempfile.mktemp(suffix=".mp3")
    client_local = OpenAI()
    resp = client_local.audio.speech.create(
        model=MODEL_TTS,
        voice=voice,
        input=sample_text
    )
    resp.stream_to_file(out_path)
    kb = os.path.getsize(out_path) // 1024
    voice_paths[voice] = out_path
    print(f"  {voice:<8}: {out_path} ({kb} KB)")

print()
print("Play the mp3 files above to compare voice styles.")
print(f"Text: '{sample_text}'")

---
## Part 13 — Pipeline Summary

Let's consolidate everything into a single clean `run_voice_pipeline()` function that mirrors the `main.py` entry point.

This is the **production-ready baseline** — three API calls, measurable latency at each stage, and a TypedDict state that makes the data flow explicit.

In [ ]:
def run_voice_pipeline(question: str) -> dict:
    """
    Full voice pipeline:
      1. TTS: question text → audio file
      2. STT: audio file → transcript
      3. LLM: transcript → text response
      4. TTS: text response → audio file

    Returns a dict with keys:
      question, audio_in, transcript, response, audio_out
    """
    # Step 0: synthesise the input audio (makes demo self-contained)
    audio_in_path = tempfile.mktemp(suffix=".mp3")
    text_to_speech(question, audio_in_path)
    in_kb = os.path.getsize(audio_in_path) // 1024
    print(f"[0] Input audio ready: {in_kb} KB")

    # Steps 1-3: run the LangGraph pipeline
    app_run = create_workflow()
    result = app_run.invoke({
        "audio_in": audio_in_path,
        "transcript": "",
        "response": "",
        "audio_out": "",
    })

    out_kb = os.path.getsize(result["audio_out"]) // 1024
    print(f"[4] Output audio ready: {out_kb} KB")

    return {
        "question":   question,
        "audio_in":   audio_in_path,
        "transcript": result["transcript"],
        "response":   result["response"],
        "audio_out":  result["audio_out"],
    }


# Run a fresh end-to-end demo
print("End-to-end voice pipeline run:")
print("=" * 60)
final_result = run_voice_pipeline(DEMO_QUESTION)
print()
print(f"Question   : {final_result['question']}")
print(f"Transcript : {final_result['transcript']}")
print(f"Response   : {final_result['response'][:200]}...")
print(f"Audio in   : {final_result['audio_in']}")
print(f"Audio out  : {final_result['audio_out']}")

---
## Exercises

Work through these exercises to deepen your understanding of the voice pipeline.

---

### Exercise 1 — Change the TTS voice

Modify `run_voice_pipeline()` to accept a `voice` parameter (default `"nova"`) and use it when generating the output audio. Run it twice with two different voices and compare the output files.

**Hint**: `text_to_speech()` calls `client.audio.speech.create(voice=TTS_VOICE, ...)`. You'll need to thread the `voice` parameter through.

---

### Exercise 2 — Add a system prompt to the LLM node

The `respond` node currently sends just a `HumanMessage`. Modify it to also send a `SystemMessage` that instructs the LLM to give concise answers suitable for voice output (under 100 words).

**Hint**: `from langchain_core.messages import SystemMessage`. Pass `[SystemMessage(content=...), HumanMessage(content=...)]` to `llm.invoke()`.

---

### Exercise 3 — Count API calls

Wrap the three API functions (`speech_to_text`, `respond`'s LLM call, `text_to_speech`) with a counter that tracks how many times each was called in a session. Print a summary at the end.

**Hint**: Use a `collections.Counter` or a simple module-level dict incremented inside each function.

In [ ]:
# ── Exercise 1 Answer: voice parameter ──────────────────────────────────────

def text_to_speech_v2(text: str, out_path: str, voice: str = "nova") -> str:
    """TTS with configurable voice."""
    client_ex = OpenAI()
    resp = client_ex.audio.speech.create(model=MODEL_TTS, voice=voice, input=text)
    resp.stream_to_file(out_path)
    return out_path


def speak_v2(state: VoiceState, voice: str = "nova") -> dict:
    """Speak node with configurable voice."""
    out = tempfile.mktemp(suffix=".mp3")
    text_to_speech_v2(state["response"], out, voice=voice)
    return {"audio_out": out}


def create_workflow_v2(voice: str = "nova"):
    """Factory that accepts a voice parameter."""
    import functools
    g = StateGraph(VoiceState)
    g.add_node("transcribe", transcribe)
    g.add_node("respond", respond)
    g.add_node("speak", functools.partial(speak_v2, voice=voice))
    g.add_edge(START, "transcribe")
    g.add_edge("transcribe", "respond")
    g.add_edge("respond", "speak")
    g.add_edge("speak", END)
    return g.compile()


# Test with "shimmer" voice
print("Exercise 1: running pipeline with 'shimmer' voice...")
ex1_app = create_workflow_v2(voice="shimmer")
ex1_result = ex1_app.invoke({
    "audio_in": audio_in,
    "transcript": "",
    "response": "",
    "audio_out": "",
})
ex1_kb = os.path.getsize(ex1_result["audio_out"]) // 1024
print(f"  Shimmer output: {ex1_result['audio_out']} ({ex1_kb} KB)")
print()

In [ ]:
# ── Exercise 2 Answer: system prompt for concise answers ────────────────────

from langchain_core.messages import SystemMessage

VOICE_SYSTEM_PROMPT = (
    "You are a helpful voice assistant. "
    "Give concise answers suitable for audio playback: "
    "under 100 words, no markdown, no bullet points, "
    "natural spoken sentences only."
)


def respond_concise(state: VoiceState) -> dict:
    """Respond node with system prompt for voice-optimised output."""
    llm = ChatOpenAI(model=MODEL_LLM)
    answer = llm.invoke([
        SystemMessage(content=VOICE_SYSTEM_PROMPT),
        HumanMessage(content=state["transcript"]),
    ]).content
    return {"response": answer}


def create_workflow_concise():
    g = StateGraph(VoiceState)
    g.add_node("transcribe", transcribe)
    g.add_node("respond", respond_concise)
    g.add_node("speak", speak)
    g.add_edge(START, "transcribe")
    g.add_edge("transcribe", "respond")
    g.add_edge("respond", "speak")
    g.add_edge("speak", END)
    return g.compile()


print("Exercise 2: concise voice-optimised response...")
ex2_app = create_workflow_concise()
ex2_result = ex2_app.invoke({
    "audio_in": audio_in,
    "transcript": "",
    "response": "",
    "audio_out": "",
})
print(f"  Response ({len(ex2_result['response'])} chars):")
print(f"  {ex2_result['response']}")

In [ ]:
# ── Exercise 3 Answer: API call counter ─────────────────────────────────────

import collections

# Module-level counter (persists across node calls within a session)
API_CALL_COUNTER = collections.Counter()


def counted_stt(audio_path: str) -> str:
    """STT with call counting."""
    API_CALL_COUNTER["stt"] += 1
    return speech_to_text(audio_path)


def counted_tts(text: str, out_path: str) -> str:
    """TTS with call counting."""
    API_CALL_COUNTER["tts"] += 1
    return text_to_speech(text, out_path)


def counted_transcribe(state: VoiceState) -> dict:
    text = counted_stt(state["audio_in"])
    print(f"  [STT] {text}")
    return {"transcript": text}


def counted_respond(state: VoiceState) -> dict:
    API_CALL_COUNTER["llm"] += 1
    llm = ChatOpenAI(model=MODEL_LLM)
    answer = llm.invoke([HumanMessage(content=state["transcript"])]).content
    return {"response": answer}


def counted_speak(state: VoiceState) -> dict:
    out = tempfile.mktemp(suffix=".mp3")
    counted_tts(state["response"], out)
    return {"audio_out": out}


def create_counted_workflow():
    g = StateGraph(VoiceState)
    g.add_node("transcribe", counted_transcribe)
    g.add_node("respond", counted_respond)
    g.add_node("speak", counted_speak)
    g.add_edge(START, "transcribe")
    g.add_edge("transcribe", "respond")
    g.add_edge("respond", "speak")
    g.add_edge("speak", END)
    return g.compile()


print("Exercise 3: running pipeline with API call counting...")
API_CALL_COUNTER.clear()
ex3_app = create_counted_workflow()
ex3_result = ex3_app.invoke({
    "audio_in": audio_in,
    "transcript": "",
    "response": "",
    "audio_out": "",
})

print()
print("API call summary:")
for api, count in sorted(API_CALL_COUNTER.items()):
    print(f"  {api:<6}: {count} call(s)")
print(f"  TOTAL : {sum(API_CALL_COUNTER.values())} call(s)")

---
## Workshop Complete

You built a full end-to-end voice pipeline with LangGraph:

| Component | API | Purpose |
|-----------|-----|---------|
| `text_to_speech()` | `tts-1` | Text → spoken mp3 |
| `speech_to_text()` | `whisper-1` | Audio → transcript |
| `respond()` node | `gpt-4o-mini` | Transcript → answer |
| `VoiceState` | TypedDict | Typed pipeline state |
| `create_workflow()` | LangGraph | Wire the three-node graph |

### Key takeaways

1. **Serial baseline first** — measure before optimising TTFA
2. **Named nodes = explicit latency** — each stage is independently measurable and swappable
3. **TypedDict state** — makes data flow readable and type-safe
4. **TTFA via streaming** — start TTS on each sentence as LLM tokens arrive
5. **One API key covers all three** — Whisper, TTS, and GPT are all under `OPENAI_API_KEY`

---

**Next: Example 103** — learn how to build agents that use external tools and make decisions about which tool to call.

```bash
python examples/103-.../main.py
```